# Tearing Simulation
This notebook covers the functionality implemented to simulate the evolution and measurement of tearing modes. In this notebook, we will cover:
1) Simulate an Initially Rotating Locked Mode (IRLM)
2) Measure the amplitude of toroidal mode numbers using the Low-N array
3) To be added: Determine m/n components of IRLM using Mirnov array, simulate an Error Field Locked Mode (EFLM), and measure the amplitude of the EFLM using the Low-N array and/or flux loops.

First, we will set up some IRLM's that are triggered and evolve using hard-coded values:

## Tearing Module

Tearing Config:
The only field in here is the modes being simulated. The modes are defined by the poloidal and toroidal mode numbers $m$ and $n$, in a list like [(m1, n1), (m2, n2), ...]. The only modes presently implemented are (2, 1), (3, 1), and (3, 2).

Tearing State:
A tearing mode has three quantities which can be evolved over time: width `W`, rotation frequency `F`, and phase `mode_phase`. Each of these quantities are 0 before the mode is triggered.

Tearing params:
The parameters determine when the mode should be triggered and how it should evolve. The two big ones are the `disruption_phase` and `rotation_phase` trajectories. `disruption_phase` represents where in the disruption process the simulation is, and goes in the order `NONE`, `TQ`, `CQ`. `rotation_phase` represents where in the mode evolution process the simulation is. For an IRLM, it goes in the order `NONE`, `SPAWN`, `ROTATING`, `DECELERATING`, and `LOCKED`, whereas for an EFLM it may go `NONE`, `LOCKED`. The `SPAWN` state is instantaneous and exists for only one time step, it represents the initial kick which gets an IRLM to begin spinning. Depending on the combinations of disruption phase and rotation phase, the mode will change in amplitude and rotation frequency differently according to hard coded values, though alternative values can passed in as parameters as well.

In [ ]:
%load_ext autoreload
%autoreload 2

from popsim.modules.tearing import Tearing, generate_disruption_phase_trajectory, generate_tearing_phase_trajectory
from popsim.simulate import make_time_base

modes = [(2, 1), (3, 2)]
tearing_config = Tearing.Config(
    modes=modes,
)

tearing_initial_state = Tearing.State(
    W={mode: 0.0 for mode in modes}, F={mode: 0.0 for mode in modes}, mode_phase={mode: 0.0 for mode in modes}
)

dt = 1e-4 / 3  # s
time_base = make_time_base(t0=0.0, t1=4.0, dt=dt)

rot_dur = 1.0
locking_dur = 0.2
trigger_time = 2.0
disrupt_time = 3.5
dur_tq_to_spike = 1e-3

tearing_params = Tearing.Params(
    rot_dur=rot_dur,  # s
    locking_dur=locking_dur,  # s
    disruption_phase=generate_disruption_phase_trajectory(disrupt_time, dur_tq_to_spike, time_base, dt),
    tearing_phase=generate_tearing_phase_trajectory(trigger_time, rot_dur, locking_dur, time_base, dt),
)

tearing_module = Tearing(config=tearing_config)

## Low-N Array Module

The Low-N array is a set of magnetic probes that are used to measure the magnitude of toroidal magnetic perturbations. Since the goal is to measure the amplitude of potentially stationary modes which may be ~1 mT (compared to the 20 T toroidal field), each probe in the array is connected to another, and the difference between them is the measurement. At the moment, the design of this array is encoded in a text file `popsim/data/tearing/lown_design.txt`, but at some point in the future it should point to the device description (or at least something more permanent).

The Low-N array config also takes into account the frequency response of the probes, which is encoded in a text file `popsim/data/tearing/21_mode_resp_data.txt`.

Finally, there is the field `reconstructed_modes`, which are what the module will return measurements for. Note that the reconstructed mode numbers are independent from the tearing modes being simulated. The probe placements in this array were optimized to measure n=[1,2,3], but you could in principle resolve up to n=7 with the present design (16 probes, (16/2)-1 = 7). If you simulate higher n modes than what is being measured, the array will still return a measurement though it will be heavily skewed.

In [ ]:
import jax

from popsim.modules.magnetic_diagnostics import LowNArray, load_lown_config

jax.config.update("jax_platforms", "cpu")

probe_connections, func_Bp_per_A = load_lown_config()

lown_array_config = LowNArray.Config(func_Bp_per_A=func_Bp_per_A, probe_connections=probe_connections, reconstructed_modes=[1, 2, 3])

lown_array_module = LowNArray(config=lown_array_config)

## Tearing Simulation

Now that we have the modules set up, we can connect them together in the `TearingSim` class and get a measurement of our modes as they are evolved in time.

In [ ]:
from popsim.simulate import SimInput, simulate
from popsim.simulators.tearing_sim.model import TearingSim

sim_config = TearingSim.Config(
    tearing_module=tearing_module,
    lown_array_module=lown_array_module,
)

sim_initial_state = TearingSim.State(tearing_state=tearing_initial_state)

sim_params = TearingSim.Params(tearing_params=tearing_params)

tearing_sim = TearingSim(config=sim_config)

sim_input = SimInput(time=time_base, initial_state=sim_initial_state, params=sim_params)

sim_xarray = simulate(tearing_sim, sim_inputs=sim_input)

In [ ]:
from popsim.visualize import visualize_time_series

visualize_time_series(sim_xarray, max_cols=2)

In [ ]:
import matplotlib.pyplot as plt

mode_widths = [sim_xarray["state.tearing_state.W.(2, 1)"], sim_xarray["state.tearing_state.W.(3, 2)"]]
rotation_frequencies = [sim_xarray["state.tearing_state.F.(2, 1)"], sim_xarray["state.tearing_state.F.(3, 2)"]]
phases = [sim_xarray["state.tearing_state.mode_phase.(2, 1)"], sim_xarray["state.tearing_state.mode_phase.(3, 2)"]]
reconstructed_magnitudes = [
    sim_xarray["output.locals.reconstructed_magnitudes.1"],
    sim_xarray["output.locals.reconstructed_magnitudes.2"],
    sim_xarray["output.locals.reconstructed_magnitudes.3"],
]
time = sim_xarray["time"].values

fig, axs = plt.subplots(2, 2, figsize=(10, 8))

for mode in range(len(mode_widths)):
    mode_width = mode_widths[mode]
    rotation_frequency = rotation_frequencies[mode]
    phase = phases[mode]
    reconstructed_magnitude = reconstructed_magnitudes[mode]
    axs[0, 0].plot(time, rotation_frequency, label=f"{modes[mode]}")
    axs[0, 1].plot(time, phase, label=f"{modes[mode]}")
    axs[1, 0].plot(time, mode_width, label=f"{modes[mode]}")
    axs[1, 1].plot(time, reconstructed_magnitude, label=f"n={mode+1}")

axs[1, 1].plot(time, reconstructed_magnitudes[2], label="n=3")

axs[0, 0].set_title("Rotation Frequency")
axs[0, 0].set_xlabel("Time [s]")
axs[0, 0].set_ylabel("Frequency [Hz]")
axs[0, 0].legend()

axs[0, 1].set_title("Mode Phase")
axs[0, 1].set_xlabel("Time [s]")
axs[0, 1].set_ylabel("Phase [rad]")
axs[0, 1].legend()

axs[1, 0].set_title("Mode Width")
axs[1, 0].set_xlabel("Time [s]")
axs[1, 0].set_ylabel("Width [m]")
axs[1, 0].legend()

axs[1, 1].set_title("Reconstructed Magnitudes")
axs[1, 1].set_xlabel("Time [s]")
axs[1, 1].set_ylabel("Magnitude [G]")
axs[1, 1].legend()


plt.tight_layout()
plt.show()